# Predicting Regional Retail Trade in Kazakhstan

## Overview

This notebook presents a linear regression analysis of monthly retail trade volume across Kazakhstan's 20 regions. The dataset was compiled from official statistics published by the Bureau of National Statistics of the Republic of Kazakhstan (stat.gov.kz).

## Problem Statement

Each month, the Bureau of National Statistics publishes a bulletin titled *"Socio-Economic Development of the Republic of Kazakhstan,"* reporting retail trade volume, inflation, household income, unemployment, and capital investment for each of the country's 20 regions, including the cities of Astana, Almaty, and Shymkent. Four of these bulletins — February 2024, June 2024, June 2025, and December 2025 — were compiled into a single panel dataset of 20 regions × 4 time periods (80 observations).

**Objective:** predict a region's monthly retail trade volume (million KZT) from five indicators — population, average income per capita, inflation, capital investment, and the unemployment rate — and quantify the contribution of each.

## Data Source

The dataset was compiled from the following official bulletins:
- [Socio-Economic Development of Kazakhstan, February 2024](https://stat.gov.kz/upload/iblock/002/iymgccgoyj2adx0lgbbu1ovcapdity11/Ж-01-01-М%2002%202024%20(рус).pdf)
- [Socio-Economic Development of Kazakhstan, June 2024](https://stat.gov.kz/upload/iblock/b16/u1k0y8ikgvnv2y0vgxzc0gg6cx4ru9f5/Ж-01-М%2006%202024%20(рус).pdf)
- [Socio-Economic Development of Kazakhstan, June 2025](https://stat.gov.kz/upload/iblock/c98/gfrc35fthdnpbtp34vzz4i4xu1dc4cz4/Ж-01-М%2006%202025%20(рус).pdf)
- [Socio-Economic Development of Kazakhstan, December 2025](https://stat.gov.kz/upload/iblock/082/mwq0xdd7o5ukymteea8y9n1kyvhg12xn/Ж-01-М%2012%202025%20(рус).pdf)

Income and unemployment are published quarterly by the source rather than monthly; each monthly observation was matched to the nearest available quarter, a standard approach when combining indicators reported at different frequencies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('../data/kazakhstan_regional_retail_trade.csv')
df['Region_label'] = df['Region'].str.replace('_', ' ')
df.head(10)

## What's in the table

| Column | Meaning |
|---|---|
| `Region` | one of Kazakhstan's 20 regions (including the three cities of republican significance) |
| `Period` | the month-year of the snapshot |
| `Retail_Trade_mln_KZT` | **target variable** — retail trade volume for that month, million KZT |
| `Population_thousand` | regional population at the start of the period, thousands |
| `Income_per_capita_KZT` | average nominal monthly income per capita, nearest quarter, KZT |
| `CPI_YoY_percent` | year-over-year consumer price inflation, % |
| `Investment_mln_KZT` | fixed capital investment for that month, million KZT |
| `Unemployment_rate_percent` | unemployment rate, nearest quarter, % |

In [ ]:
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print(f"\nRegions: {df['Region'].nunique()}")
print(f"Periods: {sorted(df['Period'].unique())}")
df.describe().round(2)

## Initial Observations

Initial histograms indicate that the raw data requires transformation before linear regression is applied. Almaty is the country's financial center with a population of 2.3 million; Ulytau is a sparsely populated region with approximately 220,000 residents. The gap in retail trade volume between the two reaches approximately 160x. A linear model fit on raw values would primarily reflect the variance contributed by a small number of large cities, rather than the underlying relationship across all 20 regions.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Feature distributions across 20 regions (4 periods, n=80)', fontsize=14, fontweight='bold')
cols = ['Retail_Trade_mln_KZT','CPI_YoY_percent','Population_thousand',
        'Investment_mln_KZT','Income_per_capita_KZT','Unemployment_rate_percent']
colors = ['#1565C0','#43A047','#FB8C00','#8E24AA','#E53935','#00897B']
for ax, col, c in zip(axes.flatten(), cols, colors):
    ax.hist(df[col], bins=20, color=c, edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=11)
plt.tight_layout()
plt.savefig('../images/1_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

`Retail_Trade`, `Population`, and `Investment` are all heavily right-skewed — a long tail driven by a handful of large cities. This is a normal pattern in regional economics: city sizes within a country tend to roughly follow a power law (Zipf's law territory), not a normal distribution.

**The fix** is to log-transform the variables that scale multiplicatively rather than additively. This isn't a cosmetic trick — it's the more honest way to model the question. "What percentage does retail trade change by if population grows 1%" is a question that makes sense everywhere, from Ulytau to Almaty. "How many million KZT does retail trade change by if population grows by a thousand people" doesn't — a thousand extra residents means something completely different in Almaty than it does in Ulytau.

In [ ]:
df['log_retail'] = np.log(df['Retail_Trade_mln_KZT'])
df['log_population'] = np.log(df['Population_thousand'])
df['log_investment'] = np.log(df['Investment_mln_KZT'])
df['log_income'] = np.log(df['Income_per_capita_KZT'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Retail_Trade before and after log transform', fontsize=13, fontweight='bold')
axes[0].hist(df['Retail_Trade_mln_KZT'], bins=25, color='#1565C0', alpha=0.85)
axes[0].set_title('Raw values (mln KZT)')
axes[1].hist(df['log_retail'], bins=25, color='#43A047', alpha=0.85)
axes[1].set_title('After log()')
plt.tight_layout()
plt.savefig('../images/1b_log_transform.png', dpi=150, bbox_inches='tight')
plt.show()

After the log transform, the distribution is substantially closer to normal, and the largest cities no longer dominate the overall variance.

## Correlations

In [ ]:
corr_df = df[['log_retail','log_population','log_income','log_investment',
              'CPI_YoY_percent','Unemployment_rate_percent']].copy()
corr_df.columns = ['Retail(log)','Population(log)','Income(log)','Investment(log)','CPI_YoY','Unemployment']
corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(8,6.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={'size':10,'fontweight':'bold'})
ax.set_title('Feature correlation (log scale)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/2_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

corr['Retail(log)'].sort_values(ascending=False)

The two strongest correlations with retail trade are investment (r≈0.67) and population (r≈0.65), driven by distinct mechanisms.

**Population** has the most direct, near-mechanical relationship. Retail trade is the aggregate of individual purchases; a larger number of residents corresponds to a larger number of transactions. The direction of causality here is largely unambiguous.

**Capital investment** is a less direct but informative signal. Investment reflects capital flows from business and government into new infrastructure — shopping centers, logistics, production facilities — rather than household spending directly. A region attracting substantial investment is typically a region with an actively growing economy, and that growth is associated with higher consumption. The relationship is likely bidirectional: investment builds infrastructure that supports retail activity, while growing retail demand attracts further investment.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(13,5.5))
fig.suptitle('Population and income per capita vs retail trade (log-log)', fontsize=13, fontweight='bold')

ax = axes[0]
ax.scatter(df['log_population'], df['log_retail'], alpha=0.6, s=40, color='#1565C0')
z = np.polyfit(df['log_population'], df['log_retail'], 1)
xline = np.linspace(df['log_population'].min(), df['log_population'].max(), 50)
ax.plot(xline, np.poly1d(z)(xline), 'r-', lw=2)
ax.set_xlabel('log(Population, thousand)')
ax.set_ylabel('log(Retail trade, mln KZT)')
ax.set_title(f"r = {corr.loc['Population(log)','Retail(log)']:.2f}")

ax = axes[1]
ax.scatter(df['log_income'], df['log_retail'], alpha=0.6, s=40, color='#E65100')
z = np.polyfit(df['log_income'], df['log_retail'], 1)
xline = np.linspace(df['log_income'].min(), df['log_income'].max(), 50)
ax.plot(xline, np.poly1d(z)(xline), 'r-', lw=2)
ax.set_xlabel('log(Income per capita, KZT/month)')
ax.set_ylabel('log(Retail trade, mln KZT)')
ax.set_title(f"r = {corr.loc['Income(log)','Retail(log)']:.2f}")

plt.tight_layout()
plt.savefig('../images/3_scatter_population_income.png', dpi=150, bbox_inches='tight')
plt.show()

**Income per capita** operates through a different mechanism than population. It reflects spending power per individual rather than the number of buyers — the disposable income available after covering basic needs. The relationship follows the standard consumption function from macroeconomic theory: higher disposable income corresponds to higher consumer spending. The right-hand chart shows Atyrau region, where average income is driven by the oil sector, positioned above the income trend line but not proportionally above it on retail trade — indicating that a substantial share of regional income does not translate into local retail spending, likely flowing into savings or investment outside the region.

## Who sells how much: regions in December 2025

In [ ]:
dec = df[df['Period']=='2025-12'].sort_values('Retail_Trade_mln_KZT', ascending=True)
fig, ax = plt.subplots(figsize=(9,9))
bars = ax.barh(dec['Region_label'], dec['Retail_Trade_mln_KZT']/1000, color='#1976D2', alpha=0.85)
bars[-1].set_color('#E53935')
bars[-2].set_color('#FB8C00')
ax.set_xlabel('Retail trade, billion KZT (December 2025)')
ax.set_title('Retail trade volume by region, December 2025', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/4_regions_dec2025.png', dpi=150, bbox_inches='tight')
plt.show()

Almaty (red) records retail trade approximately 2.3x higher than Astana (orange), despite a comparatively smaller gap in population (2.3 million vs. 1.6 million). This reflects Almaty's established role as the country's primary commercial center, a structural feature of Kazakhstan's economy that the regression model also captures.

## What's going on with inflation

In [ ]:
period_avg = df.groupby('Period')[['CPI_YoY_percent','Retail_Trade_mln_KZT']].mean()
fig, ax1 = plt.subplots(figsize=(9,5.5))
ax2 = ax1.twinx()
ax1.plot(period_avg.index, period_avg['CPI_YoY_percent'], 'o-', color='#C62828', linewidth=2, label='Inflation (CPI YoY, %)')
ax2.plot(period_avg.index, period_avg['Retail_Trade_mln_KZT']/1000, 's-', color='#1565C0', linewidth=2, label='Average retail trade (bln KZT)')
ax1.set_ylabel('CPI year-over-year, %', color='#C62828')
ax2.set_ylabel('Retail trade, billion KZT', color='#1565C0')
ax1.set_title('Inflation and average regional sales over time', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/5_cpi_trend.png', dpi=150, bbox_inches='tight')
plt.show()

The direction of causality should not be misread here. Inflation in Kazakhstan remained persistently elevated over this period (109–113% year-over-year), and nominal retail trade nearly doubled over the same two years. This does not imply that consumption volume doubled — retail trade is reported in current, not inflation-adjusted, prices. Part of the increase reflects genuine growth in consumption; part reflects the same goods costing more in 2025 than in 2024. The model does not separate these two effects, which is addressed explicitly in the limitations section below.

## The model

Features: `log_population`, `log_income`, `CPI_YoY_percent`, `log_investment`, `Unemployment_rate_percent`.
Target: `log_retail`.

75/25 split, `random_state=42`.

In [ ]:
feature_cols = ['log_population','log_income','CPI_YoY_percent','log_investment','Unemployment_rate_percent']
target_col = 'log_retail'

X = df[feature_cols].copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Train: {len(X_train)} rows, Test: {len(X_test)} rows")

model = LinearRegression()
model.fit(X_train, y_train)
pred_train = model.predict(X_train)
pred_test = model.predict(X_test)
print("Fitted")

## Metrics

R² is used as the primary metric for comparing train and test performance (MAE and RMSE are not directly comparable to one another — they are based on different error formulations), with MAE and RMSE reported as well to indicate error magnitude in concrete units.

In [ ]:
def show_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: MAE={mae:.4f} | RMSE={rmse:.4f} | R²={r2:.4f}  (log units)")
    return mae, rmse, r2

mae_tr, rmse_tr, r2_tr = show_metrics(y_train, pred_train, "TRAIN")
mae_te, rmse_te, r2_te = show_metrics(y_test, pred_test, "TEST ")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring='r2')
print(f"\n5-fold CV R²: mean={cv_scores.mean():.3f}, std={cv_scores.std():.3f}")
print(f"Per fold: {np.round(cv_scores,3)}")

pred_test_real = np.exp(pred_test)
y_test_real = np.exp(y_test)
mae_real = mean_absolute_error(y_test_real, pred_test_real)
mape = np.mean(np.abs((y_test_real - pred_test_real) / y_test_real)) * 100
print(f"\nIn real KZT: average error ≈ {mae_real:,.0f} mln KZT, MAPE ≈ {mape:.0f}%")

Test R² is 0.72: the model explains approximately 72% of the variance in log(retail trade) using five features, a reasonable result for 80 observations of regional economic data with inherent noise. The 5-fold CV scores vary substantially (0.05 to 0.88), which is expected given the small effective sample size: the same region appears across multiple periods, and when an atypical case (Atyrau, with its oil-driven income, is a relevant example) falls into the test fold, R² on that fold declines.

A MAPE of approximately 40% reflects the model's ability to estimate the correct order of magnitude across a 160x size range between regions, rather than precise totals — an expected ceiling for a linear model fit on 80 observations.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,5.5))
fig.suptitle('Model performance (back-transformed to real KZT)', fontsize=13, fontweight='bold')

ax = axes[0]
ax.scatter(y_test_real/1000, pred_test_real/1000, alpha=0.7, s=60, color='#1565C0')
lims = [0, max(y_test_real.max(), pred_test_real.max())/1000*1.1]
ax.plot(lims, lims, 'r--', lw=2)
ax.set_xlabel('Actual volume (bln KZT)')
ax.set_ylabel('Predicted volume (bln KZT)')
ax.set_title(f'Actual vs Predicted | R²={r2_te:.3f}')

residuals = y_test - pred_test
ax = axes[1]
ax.hist(residuals, bins=12, color='#EF5350', edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', lw=2, linestyle='--')
ax.set_xlabel('Residual (log units)')
ax.set_title('Residual distribution')

plt.tight_layout()
plt.savefig('../images/6_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## Coefficient Interpretation

Because the model is specified in logs for four of the five features (`log_population`, `log_income`, `log_investment`, and the target `log_retail`), the corresponding coefficients are **elasticities**: the percentage change in retail trade associated with a 1% change in the feature. CPI and unemployment remain in their original percentage-point units, so their coefficients are read as the absolute change in log(retail trade) per one additional percentage point.

In [ ]:
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': model.coef_}).sort_values('Coefficient', key=abs)
print(coef_df.to_string(index=False))
print(f"\nIntercept: {model.intercept_:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9,5.5))
colors_bar = ['#1565C0' if c>0 else '#E53935' for c in coef_df['Coefficient']]
bars = ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_bar, alpha=0.9)
ax.axvline(0, color='black', lw=1.2)
ax.set_xlabel('Coefficient (elasticity for log-transformed features)')
ax.set_title('Contribution of each feature to the model', fontsize=13, fontweight='bold')
for bar, val in zip(bars, coef_df['Coefficient']):
    ax.text(val + (0.02 if val>=0 else -0.02), bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val>=0 else 'right', fontsize=10)
plt.tight_layout()
plt.savefig('../images/7_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

**`log_income` ≈ 1.53** — the strongest effect in the model, and notably above 1: a 1% increase in income is associated with retail trade growing by more than 1%. An income elasticity above one is consistent with discretionary spending (dining, electronics, apparel) growing disproportionately faster than spending on necessities as income rises.

**`log_population` ≈ 1.22** — also above 1, which is a less intuitive result. If each resident purchased an identical fixed basket of goods, the elasticity would equal exactly 1. An elasticity above 1 indicates that larger cities generate disproportionately higher retail trade per capita, consistent with Almaty and Astana functioning as retail and logistics hubs that draw consumption from neighboring, less populated regions, not only from their own residents.

**`Unemployment` ≈ −0.44** — exponentiating this coefficient (exp(−0.437) ≈ 0.65) indicates that a one-percentage-point increase in unemployment is associated with approximately 35% lower retail trade, holding other variables constant. This is the only feature with a negative coefficient in the model, consistent with the direct mechanism: reduced employment corresponds to reduced household income and reduced spending.

**`log_investment` ≈ 0.11** — the weakest of the meaningful effects. Fixed capital investment (factories, roads, infrastructure) typically affects consumption with a lag of months to years, through job creation and infrastructure development, rather than instantaneously within the same reporting month.

**`CPI_YoY` ≈ 0.02** — positive but small. Exponentiated, each additional percentage point of inflation is associated with approximately a 2.3% increase in nominal retail trade, holding other variables constant. This is consistent with the inflation trend discussed earlier: most of the headline inflation effect on nominal sales is already captured by the time trend between periods, leaving a modest standalone CPI effect once other variables are controlled for.

## Limitations

**Effective sample size.** The dataset contains 80 observations, representing 20 regions observed at 4 time points rather than 80 independent observations. Successive snapshots of the same region are correlated, reducing the effective sample size to closer to 20. This is reflected directly in the cross-validation spread (0.05 to 0.88 across folds).

**Nominal, not real, prices.** Retail trade is reported in current prices. The model does not separate inflation-driven nominal growth from genuine growth in consumption volume. Deflating retail trade by CPI prior to modeling would address this.

**Linear specification.** The model assumes a constant effect of each feature across all regions — for example, that the elasticity of retail trade with respect to income is identical in Almaty and in a predominantly agricultural region such as Turkestan, despite differing spending structures between metropolitan and rural regions.

**No explicit seasonality control.** The four snapshots span different months (February, June, December). December is typically associated with elevated consumer spending, which may be influencing the estimated coefficients, particularly for CPI and investment.

**Limited variation in unemployment.** Regional unemployment in Kazakhstan ranges narrowly from 4.0% to 5.1%, limiting the model's ability to estimate the magnitude of this effect with precision. The true effect may be larger than indicated by the available variation in the data.

## Suggested Extensions

Extending the time series to monthly observations over a longer period would increase the effective sample size and reduce sensitivity to any single period. Deflating retail trade by CPI would isolate real consumption growth from price effects. Including regional fixed effects (dummy variables) would account for structural differences between cities and rural oblasts. Comparing the linear specification against a non-linear model, such as Random Forest, would clarify the extent to which the underlying relationships are linear.